# Download Planet.com orders + Sentinel-2 before images

**Purpose:** Downloads Planet order results (after images) AND Sentinel-2 before images
from Google Earth Engine for the specified incident index range. The output follows
the same folder structure as `extracting_data.ipynb`:

```
downloads/
  incident_<ID>/
    incident_<ID>_after.tif     (from Planet.com order, clipped to AOI)
    incident_<ID>_before.tif    (from GEE Sentinel-2, 13 bands, 10 m)
```

**Synced with `create_planet_orders.ipynb`:** Set the same `start_idx` / `end_idx`
here as you used when creating orders. Leave both as `0` to download **all**
successful Planet orders.

This notebook runs on **Kaggle**.

## Setup (one time)
1. Get your Planet API key from https://www.planet.com/account/#/
2. In Kaggle: **Add-ons → Secrets** → add a secret named `planet_api_key`
3. GEE authentication uses the same service account key as `extracting_data.ipynb`
   (via Kaggle dataset secret). If unavailable, before-image download is skipped.

In [ ]:
# --------------------------------------------------------------------
# Imports
# --------------------------------------------------------------------
import os, re, time, glob, shutil
import pandas as pd
import requests
from requests.adapters import HTTPAdapter, Retry
from concurrent.futures import ThreadPoolExecutor, as_completed
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi

In [ ]:
# --------------------------------------------------------------------
# Configuration — edit these before running
# --------------------------------------------------------------------

# CSV containing landslide incidents (same as extracting_data.ipynb)
input_csv = "/kaggle/input/datasets/sanjayashrestha123/landslide-reproted/landslides_from_2018_to_2026.csv"

# Index range into the CSV (df.iloc[start_idx:end_idx])
# Set both to 0 to download ALL successful Planet orders (no before images).
start_idx = 0
end_idx   = 0

# Download root — matches extracting_data.ipynb structure
DOWNLOAD_DIR = "/kaggle/working/downloads"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# --- HuggingFace upload ---
repo_id = "sasudo2/landslides"
DATASET_REVISION = "main"
upload_batch = 50           # upload to HF every N incidents

# --- Planet ---
ORDERS_URL = "https://api.planet.com/compute/ops/orders/v2"
WANTED_STATES = {"success", "partial"}
PLANET_WORKERS = 4
CHUNK_SIZE = 1 << 20
REQUEST_TIMEOUT = 300

# --- GEE Sentinel-2 before image ---
# GEE project & credentials (same as extracting_data.ipynb)
GEE_PROJECT = "landslide-identification-nepal"
GEE_SERVICE_ACCOUNT = "kaggle-import@landslide-identification-nepal.iam.gserviceaccount.com"
GEE_KEY_PATH = "/kaggle/input/datasets/sanjayashrestha123/gee-key/landslide-identification-nepal-cccd90850069.json"

# AOI clamping (same as extracting_data.ipynb)
MAX_AOI_DEG = 0.1

# Sentinel-2 scene selection (tight cloud threshold for "almost no cloud")
S2_BANDS = ['B1','B2','B3','B4','B5','B6','B7','B8','B8A','B9','B11','B12','SCL']
PRE_DAYS = 180            # search window before incident
PRE_BUFFER_DAYS = 5       # exclude scenes too close to incident date
MAX_TILE_CLOUD_PCT = 50   # loose whole-tile prefilter
MAX_AOI_CLOUD_PCT = 10    # strict AOI-level cloud ("almost no cloud")
GEE_SCALE = 10            # 10 m for S2
GEE_MAX_RETRIES = 5

# --- Sentinel-1 SAR ---
SAR_PRE_DAYS = 45         # SAR pre-event window (days before incident)
SAR_POST_DAYS = 45        # SAR post-event window (days after incident)

# --- GEE availability flag (set after auth) ---
gee_available = False

In [ ]:
# --------------------------------------------------------------------
# Authenticate: HuggingFace
# --------------------------------------------------------------------
user_secrets = UserSecretsClient()
huggingface_key = user_secrets.get_secret("huggingface_token")
hf_api = HfApi(token=huggingface_key)
hf_api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
print(f"HuggingFace repo '{repo_id}' ready.")

# --------------------------------------------------------------------
# Authenticate: Planet
# --------------------------------------------------------------------
PLANET_API_KEY = user_secrets.get_secret("planet_api_key")

def make_planet_session():
    s = requests.Session()
    s.auth = (PLANET_API_KEY, "")
    retries = Retry(
        total=5,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        respect_retry_after_header=True,
    )
    s.mount("https://", HTTPAdapter(max_retries=retries, pool_maxsize=PLANET_WORKERS * 2))
    return s

session = make_planet_session()
r = session.get(ORDERS_URL, timeout=60)
r.raise_for_status()
print("Planet authentication OK.")

# --------------------------------------------------------------------
# Authenticate: Google Earth Engine (graceful failure)
# --------------------------------------------------------------------
try:
    import ee
    credentials = ee.ServiceAccountCredentials(GEE_SERVICE_ACCOUNT, GEE_KEY_PATH)
    ee.Initialize(credentials, project=GEE_PROJECT)
    gee_available = True
    print("Google Earth Engine authenticated.")
except Exception as e:
    print(f"GEE init failed — before-image download will be skipped: {e}")

In [ ]:
# --------------------------------------------------------------------
# Upload helper — mirrors extracting_data.ipynb
# --------------------------------------------------------------------
upload_count = 0

def flush_uploads():
    global upload_count
    tif_count = len(glob.glob(f"{DOWNLOAD_DIR}/**/*.tif", recursive=True))
    if tif_count == 0:
        print("No new GeoTIFFs to upload.")
        return
    print(f"Uploading {tif_count} GeoTIFFs to {repo_id}...")
    try:
        hf_api.upload_folder(
            folder_path=DOWNLOAD_DIR,
            repo_id=repo_id,
            repo_type="dataset",
            revision=DATASET_REVISION,
            allow_patterns="*.tif",
        )
        print(f"Upload successful. Cleaning up...")
        for subdir in glob.glob(f"{DOWNLOAD_DIR}/*/"):
            shutil.rmtree(subdir)
        upload_count = 0
        print(f"Cleaned {DOWNLOAD_DIR}")
    except Exception as e:
        print(f"!!! Upload failed, keeping local files for retry: {e} !!!")


In [ ]:
# --------------------------------------------------------------------
# Load the landslide incidents CSV and apply index range
# --------------------------------------------------------------------
df_csv = pd.read_csv(input_csv)
if start_idx == 0 and end_idx == 0:
    print("Index range not set (start_idx=end_idx=0) — no CSV filter applied.")
    wanted_ids = None
else:
    df_range = df_csv.iloc[start_idx:end_idx]
    df_range['incident_on'] = pd.to_datetime(df_range['incident_on'], dayfirst=True)
    wanted_ids = set(df_range['id'].astype(int).tolist())
    print(f"Loaded {len(wanted_ids)} incidents from CSV rows {start_idx}:{end_idx}.")

In [ ]:
# --------------------------------------------------------------------
# Check HF repo for existing images — avoid re-downloading
# --------------------------------------------------------------------
print("Checking HuggingFace repo for existing images...")
existing_after_ids = set()
existing_before_ids = set()
existing_slope_ids = set()
existing_aspect_ids = set()
existing_sar_pre_ids = set()
existing_sar_post_ids = set()
try:
    for f in hf_api.list_repo_files(repo_id, repo_type="dataset"):
        m = re.match(r'^incident_(\d+)/', f)
        if m:
            inc_id = int(m.group(1))
            if f.endswith('_after.tif'):
                existing_after_ids.add(inc_id)
            elif f.endswith('_before.tif'):
                existing_before_ids.add(inc_id)
            elif f.endswith('_slope.tif'):
                existing_slope_ids.add(inc_id)
            elif f.endswith('_aspect.tif'):
                existing_aspect_ids.add(inc_id)
            elif f.endswith('_sar_pre.tif'):
                existing_sar_pre_ids.add(inc_id)
            elif f.endswith('_sar_post.tif'):
                existing_sar_post_ids.add(inc_id)
    print(f"  _after.tif:    {len(existing_after_ids)} incidents")
    print(f"  _before.tif:   {len(existing_before_ids)} incidents")
    print(f"  _slope.tif:    {len(existing_slope_ids)} incidents")
    print(f"  _aspect.tif:   {len(existing_aspect_ids)} incidents")
    print(f"  _sar_pre.tif:  {len(existing_sar_pre_ids)} incidents")
    print(f"  _sar_post.tif: {len(existing_sar_post_ids)} incidents")
except Exception as e:
    print(f"  Could not query HF repo: {e}")
    print("  Proceeding without skipping.")


In [ ]:
# --------------------------------------------------------------------
# List all Planet orders, filter to index range
# --------------------------------------------------------------------
def list_orders(session):
    orders = []
    url = ORDERS_URL
    page = 0
    while url:
        resp = session.get(url, timeout=120)
        resp.raise_for_status()
        data = resp.json()
        batch = data.get("orders", [])
        orders.extend(batch)
        page += 1
        print(f"Page {page}: fetched {len(batch)} orders (total {len(orders)})")
        url = data.get("_links", {}).get("_next")
    return orders

all_orders = list_orders(session)
print(f"\nTotal orders found: {len(all_orders)}")

from collections import Counter
states = Counter(o.get("state") for o in all_orders)
print("Order states:", dict(states))

downloadable = [o for o in all_orders if o.get("state") in WANTED_STATES]
print(f"Downloadable orders ({'/'.join(sorted(WANTED_STATES))}): {len(downloadable)}")

pattern = re.compile(r'^incident_(\d+)_after$')
before = len(downloadable)
downloadable = [
    o for o in downloadable
    if (m := pattern.match(o.get("name", "")))
    and (wanted_ids is None or int(m.group(1)) in wanted_ids)
    and int(m.group(1)) not in existing_after_ids
]
print(f"Filtered: {before} → {len(downloadable)} matching orders (CSV range + not in HF).")

In [ ]:
# --------------------------------------------------------------------
# Download Planet order results as incident_<ID>_after.tif
# --------------------------------------------------------------------
def get_order_results(session, order):
    results = order.get("_links", {}).get("results")
    if results:
        return results
    order_id = order.get("id")
    detail = session.get(f"{ORDERS_URL}/{order_id}", timeout=120)
    detail.raise_for_status()
    return detail.json().get("_links", {}).get("results", []) or []

def download_planet_result(order):
    order_name = order.get("name", "")
    m = re.match(r'^incident_(\d+)_after$', order_name)
    if not m:
        return ("skip", f"{order_name}: name does not match pattern")
    incident_id = m.group(1)
    results = get_order_results(session, order)
    if not results:
        return ("skip", f"incident_{incident_id}: no results in order")

    # Save as incident_<ID>/incident_<ID>_after.tif  (extracting_data structure)
    folder = os.path.join(DOWNLOAD_DIR, f"incident_{incident_id}")
    local_path = os.path.join(folder, f"incident_{incident_id}_after.tif")

    if os.path.exists(local_path) and os.path.getsize(local_path) > 0:
        return ("skip", local_path)

    os.makedirs(folder, exist_ok=True)
    url = results[0].get("location")
    if not url:
        return ("error", f"incident_{incident_id}: no download URL")

    tmp_path = local_path + ".part"
    try:
        with session.get(url, stream=True, timeout=REQUEST_TIMEOUT) as r:
            r.raise_for_status()
            with open(tmp_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                    if chunk:
                        f.write(chunk)
        os.replace(tmp_path, local_path)
        return ("ok", local_path)
    except Exception as e:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        return ("error", f"incident_{incident_id}: {e}")

planet_ok = planet_skip = planet_fail = 0
planet_errors = []
with ThreadPoolExecutor(max_workers=PLANET_WORKERS) as ex:
    futures = {ex.submit(download_planet_result, o): o for o in downloadable}
    for i, fut in enumerate(as_completed(futures), 1):
        status, info = fut.result()
        if status == "ok":
            planet_ok += 1
        elif status == "skip":
            planet_skip += 1
        else:
            planet_fail += 1
            planet_errors.append(info)
        if i % 25 == 0 or i == len(futures):
            print(f"[Planet {i}/{len(futures)}] ok={planet_ok} skip={planet_skip} fail={planet_fail}")

print(f"\nPlanet downloads: {planet_ok} ok, {planet_skip} skipped, {planet_fail} failed.")
if planet_errors:
    for e in planet_errors[:5]:
        print(f"  - {e}")

upload_count += planet_ok
if upload_count >= upload_batch:
    flush_uploads()

## GEE downloads: before S2 + DEM + SAR

Mirrors `extracting_data.ipynb`: downloads Sentinel-2 before image (13 bands,
10 m), DEM slope/aspect (30 m), and Sentinel-1 GRD pre/post (VV/VH, 10 m) for
each incident. Cloud threshold is stricter (`MAX_AOI_CLOUD_PCT = 10`).

In [ ]:
# --------------------------------------------------------------------
# GEE helper functions (identical to extracting_data.ipynb)
# --------------------------------------------------------------------
def mask_s2_clouds(image):
    scl = image.select('SCL')
    clean_mask = (scl.eq(2).bitwiseOr(scl.eq(4))
                           .bitwiseOr(scl.eq(5))
                           .bitwiseOr(scl.eq(6))
                           .bitwiseOr(scl.eq(7))
                           .bitwiseOr(scl.eq(11)))
    return image.updateMask(clean_mask)


def add_aoi_cloud(img, aoi):
    scl = img.select('SCL')
    cloud = (scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10)))
    stats = cloud.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=60,
        maxPixels=1e9,
    )
    frac = stats.get('SCL')
    pct = ee.Algorithms.If(frac, ee.Number(frac).multiply(100), ee.Number(100))
    return img.set('aoi_cloud', pct)


def clamp_aoi(min_lon, min_lat, max_lon, max_lat):
    lon_span = max_lon - min_lon
    lat_span = max_lat - min_lat
    if lon_span <= MAX_AOI_DEG and lat_span <= MAX_AOI_DEG:
        return min_lon, min_lat, max_lon, max_lat
    cx = (min_lon + max_lon) / 2
    cy = (min_lat + max_lat) / 2
    half = MAX_AOI_DEG / 2
    return cx - half, cy - half, cx + half, cy + half


def download_gee_image(image, aoi, incident_id, filename, scale=GEE_SCALE):
    folder = f"{DOWNLOAD_DIR}/incident_{incident_id}"
    os.makedirs(folder, exist_ok=True)
    filepath = f'{folder}/{filename}.tif'
    if os.path.exists(filepath) and os.path.getsize(filepath) > 0:
        print(f"    Already exists: {filepath}")
        return
    for attempt in range(1, GEE_MAX_RETRIES + 1):
        try:
            url = image.getDownloadURL({
                'scale': scale,
                'region': aoi,
                'format': 'GeoTIFF',
                'crs': 'EPSG:4326',
            })
            resp = requests.get(url, stream=True, timeout=300)
            if resp.status_code == 429:
                wait = 15 * attempt
                print(f"    429 on {filename}, retry {attempt}/{GEE_MAX_RETRIES} after {wait}s")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            with open(filepath, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"    Downloaded: {filepath}")
            return
        except Exception as e:
            if attempt == GEE_MAX_RETRIES:
                print(f"    Failed to download {filename}: {e}")
                return
            wait = 15 * attempt
            print(f"    error on {filename}, retry {attempt}/{GEE_MAX_RETRIES} after {wait}s: {e}")
            time.sleep(wait)


def pick_best_scene(collection, label, reverse=False):
    count = collection.size().getInfo()
    if count == 0:
        print(f"    {label}: no scenes found.")
        return None
    clear = collection.filter(ee.Filter.lte('aoi_cloud', MAX_AOI_CLOUD_PCT))
    if clear.size().getInfo() > 0:
        return clear.sort('system:time_start', reverse).first()
    print(f"    {label}: no scene ≤ {MAX_AOI_CLOUD_PCT}% AOI cloud; using least-cloudy.")
    return collection.sort('aoi_cloud').first()


def pick_closest_scene(collection, label, reverse=False):
    count = collection.size().getInfo()
    if count == 0:
        print(f"    {label}: no scenes found.")
        return None
    return collection.sort('system:time_start', reverse).first()

In [ ]:
# --------------------------------------------------------------------
# Download GEE data: S2 before + DEM slope/aspect + Sentinel-1 SAR
# --------------------------------------------------------------------
if not gee_available:
    print("\nGEE not available — skipping GEE downloads.")
elif wanted_ids is None:
    print("\nNo index range set — skipping GEE downloads (no CSV context).")
else:
    print(f"\n=== GEE downloads for {len(wanted_ids)} incidents ===\n")
    gee_downloaded = 0

    for inc_id in sorted(wanted_ids):
        row = df_csv[df_csv['id'] == inc_id]
        if row.empty:
            continue
        row = row.iloc[0]
        incident_date = pd.to_datetime(row['incident_on'], dayfirst=True)

        c_min_lon, c_min_lat, c_max_lon, c_max_lat = clamp_aoi(
            row['min_lon'], row['min_lat'], row['max_lon'], row['max_lat'])
        aoi = ee.Geometry.Rectangle([c_min_lon, c_min_lat, c_max_lon, c_max_lat])

        before_start = (incident_date - pd.DateOffset(days=PRE_DAYS)).strftime('%Y-%m-%d')
        before_end   = (incident_date - pd.DateOffset(days=PRE_BUFFER_DAYS)).strftime('%Y-%m-%d')
        after_start  = (incident_date + pd.DateOffset(days=PRE_BUFFER_DAYS)).strftime('%Y-%m-%d')

        print(f"\nIncident {inc_id}: {row['title']}")

        # --- Sentinel-2 before (13 bands, 10 m) ---
        if inc_id not in existing_before_ids:
            before_path = f"{DOWNLOAD_DIR}/incident_{inc_id}/incident_{inc_id}_before.tif"
            if os.path.exists(before_path) and os.path.getsize(before_path) > 0:
                print(f"  Before image already on disk.")
            else:
                s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                        .filterBounds(aoi)
                        .filter(ee.Filter.lte('CLOUDY_PIXEL_PERCENTAGE', MAX_TILE_CLOUD_PCT))
                        .map(lambda img: add_aoi_cloud(img, aoi)))
                before_img = pick_best_scene(
                    s2.filterDate(before_start, before_end).map(mask_s2_clouds),
                    'before', reverse=True)
                if before_img:
                    print(f"  Before date: {before_img.date().format().getInfo()}")
                    download_gee_image(
                        before_img.select(S2_BANDS).clip(aoi),
                        aoi, inc_id, f'incident_{inc_id}_before', scale=GEE_SCALE)
        else:
            print(f"  Before image already in HF.")

        # --- DEM slope & aspect (30 m) ---
        if inc_id not in existing_slope_ids or inc_id not in existing_aspect_ids:
            dem = ee.Image('USGS/SRTMGL1_003')
            if inc_id not in existing_slope_ids:
                download_gee_image(
                    ee.Terrain.slope(dem).clip(aoi),
                    aoi, inc_id, f'incident_{inc_id}_slope', scale=30)
            if inc_id not in existing_aspect_ids:
                download_gee_image(
                    ee.Terrain.aspect(dem).clip(aoi),
                    aoi, inc_id, f'incident_{inc_id}_aspect', scale=30)
        else:
            print(f"  Slope & aspect already in HF.")

        # --- Sentinel-1 GRD pre/post (VV/VH, 10 m, same orbit) ---
        if inc_id not in existing_sar_pre_ids or inc_id not in existing_sar_post_ids:
            try:
                s1_base = (ee.ImageCollection('COPERNICUS/S1_GRD')
                            .filterBounds(aoi)
                            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
                            .filter(ee.Filter.eq('instrumentMode', 'IW')))
                sar_pre_start = (incident_date - pd.DateOffset(days=SAR_PRE_DAYS)).strftime('%Y-%m-%d')
                sar_post_end  = (incident_date + pd.DateOffset(days=SAR_POST_DAYS)).strftime('%Y-%m-%d')

                s1_pre  = s1_base.filterDate(sar_pre_start, before_end)
                s1_post = s1_base.filterDate(after_start, sar_post_end)

                pre_pass  = s1_pre.aggregate_array('orbitProperties_pass').getInfo()
                post_pass = s1_post.aggregate_array('orbitProperties_pass').getInfo()
                common_passes = set(pre_pass) & set(post_pass)
                if not common_passes:
                    raise ValueError('No common orbit pass between pre and post S1')
                common_pass = sorted(common_passes)[0]

                if inc_id not in existing_sar_pre_ids:
                    s1_img_pre = pick_closest_scene(
                        s1_pre.filter(ee.Filter.eq('orbitProperties_pass', common_pass)),
                        'SAR pre', reverse=True)
                    if s1_img_pre:
                        print(f"  SAR pre date: {s1_img_pre.date().format().getInfo()}")
                        download_gee_image(
                            s1_img_pre.select(['VV', 'VH']).clip(aoi),
                            aoi, inc_id, f'incident_{inc_id}_sar_pre', scale=10)

                if inc_id not in existing_sar_post_ids:
                    s1_img_post = pick_closest_scene(
                        s1_post.filter(ee.Filter.eq('orbitProperties_pass', common_pass)),
                        'SAR post')
                    if s1_img_post:
                        print(f"  SAR post date: {s1_img_post.date().format().getInfo()}")
                        download_gee_image(
                            s1_img_post.select(['VV', 'VH']).clip(aoi),
                            aoi, inc_id, f'incident_{inc_id}_sar_post', scale=10)
            except Exception as e:
                print(f"  SAR fetch skipped: {e}")
        else:
            print(f"  SAR pre & post already in HF.")

        gee_downloaded += 1

    print(f"\nGEE processed {gee_downloaded} incidents.")

    upload_count += gee_downloaded
    flush_uploads()

In [ ]:
# --------------------------------------------------------------------
# Summary of all downloaded data
# --------------------------------------------------------------------
total_files = 0
total_bytes = 0
for root, _, files in os.walk(DOWNLOAD_DIR):
    for fn in files:
        if fn.endswith(".part"):
            continue
        total_files += 1
        total_bytes += os.path.getsize(os.path.join(root, fn))

print(f"\n{'='*40}")
print(f"Files on disk: {total_files}")
print(f"Total size:    {total_bytes / (1024**3):.2f} GiB")
print(f"Location:      {DOWNLOAD_DIR}")

# List incident folders
incident_dirs = sorted(d for d in os.listdir(DOWNLOAD_DIR)
                      if d.startswith("incident_") and os.path.isdir(os.path.join(DOWNLOAD_DIR, d)))
if incident_dirs:
    print(f"Incident folders: {len(incident_dirs)}")
    for d in incident_dirs[:10]:
        files = [f for f in os.listdir(os.path.join(DOWNLOAD_DIR, d)) if f.endswith('.tif')]
        print(f"  {d}/: {files}")
    if len(incident_dirs) > 10:
        print(f"  ... and {len(incident_dirs) - 10} more")